In [1]:
import pandas as pd
up_wide = pd.read_parquet("../data/petrochem/up_wide.parquet")

In [2]:
product_tags = [
    'FIC10014_PV', 'FIC20017_PV', 'FIC20018_PV', 'FIC25021_PV', 
    'FIC30018_PV', 'FI40039_PV', 'FIC40045_PV', 'FIC50011_PV',
    'FIC60020_PV', 'FIC65004_PV', 'FIC70053_PV', 'FIC95008_PV'
]

In [3]:
utility_tags = [tag for tag in up_wide.columns if tag not in product_tags]

print(f"\nProduct Tags: {len([t for t in product_tags if t in up_wide.columns])}")
print(f"Utility Tags: {len(utility_tags)}\n")


Product Tags: 12
Utility Tags: 20



In [4]:
# up_wide.head()

In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

### 1. DATA PREPARATION

In [6]:
X = up_wide[utility_tags].values
y = up_wide[product_tags].values

In [7]:
valid_idx = (~np.isnan(X).any(axis=1)) & (~np.isnan(y).any(axis=1))
X = X[valid_idx]
y = y[valid_idx]

In [8]:
print(f"  Data shape: {X.shape}")
print(f"  Data shape: {y.shape}")


  Data shape: (24608, 20)
  Data shape: (24608, 12)


In [9]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)

print(f"  Train: {X_train.shape[0]}, Test: {X_test.shape[0]} ✓")

  Train: 19686, Test: 4922 ✓


### 2. Train XGBoost Models

In [10]:
models = {}
metrics_list = []

In [11]:
for prod_idx, prod_tag in enumerate(product_tags):
    print(f"Training {prod_tag}...", end=" ")
    
    # Train XGBoost
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    
    model.fit(X_train, y_train[:, prod_idx], verbose=False)
    
    # Predictions
    y_pred_test = model.predict(X_test)
    y_test_prod = y_test[:, prod_idx]
    
    # Metrics
    r2 = r2_score(y_test_prod, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test_prod, y_pred_test))
    mae = mean_absolute_error(y_test_prod, y_pred_test)
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    models[prod_tag] = model
    metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })
    
    print(f"R²: {r2:.4f} {quality}")


Training FIC10014_PV... R²: 0.2491 ⚠ Fair
Training FIC20017_PV... R²: 0.6701 ⚠ Fair
Training FIC20018_PV... R²: 0.5652 ⚠ Fair
Training FIC25021_PV... R²: 0.7531 ✓ Good
Training FIC30018_PV... R²: 0.9364 ⭐ Excellent
Training FI40039_PV... R²: 0.0201 ⚠ Fair
Training FIC40045_PV... R²: 0.6631 ⚠ Fair
Training FIC50011_PV... R²: 0.9209 ⭐ Excellent
Training FIC60020_PV... R²: 0.8964 ⭐ Excellent
Training FIC65004_PV... R²: 0.8786 ⭐ Excellent
Training FIC70053_PV... R²: 0.9087 ⭐ Excellent
Training FIC95008_PV... R²: 0.8423 ✓ Good


In [12]:
metrics_df = pd.DataFrame(metrics_list)

In [13]:
print("\nPER-PRODUCT METRICS:")
for _, row in metrics_df.iterrows():
    print(f"{row['Product']:20} | R²: {row['R²']:7.4f} | RMSE: {row['RMSE']:8.4f} | {row['Quality']}")


PER-PRODUCT METRICS:
FIC10014_PV          | R²:  0.2491 | RMSE:   0.9402 | ⚠ Fair
FIC20017_PV          | R²:  0.6701 | RMSE:   0.5523 | ⚠ Fair
FIC20018_PV          | R²:  0.5652 | RMSE:   0.6386 | ⚠ Fair
FIC25021_PV          | R²:  0.7531 | RMSE:   0.4804 | ✓ Good
FIC30018_PV          | R²:  0.9364 | RMSE:   0.2454 | ⭐ Excellent
FI40039_PV           | R²:  0.0201 | RMSE:   1.0673 | ⚠ Fair
FIC40045_PV          | R²:  0.6631 | RMSE:   0.5714 | ⚠ Fair
FIC50011_PV          | R²:  0.9209 | RMSE:   0.2768 | ⭐ Excellent
FIC60020_PV          | R²:  0.8964 | RMSE:   0.3210 | ⭐ Excellent
FIC65004_PV          | R²:  0.8786 | RMSE:   0.3465 | ⭐ Excellent
FIC70053_PV          | R²:  0.9087 | RMSE:   0.3008 | ⭐ Excellent
FIC95008_PV          | R²:  0.8423 | RMSE:   0.3906 | ✓ Good


In [32]:
overall_r2 = metrics_df['R²'].mean()
print(f"\nOverall R² Average: {overall_r2:.4f}")
print("="*120)


Overall R² Average: 0.6920


In [33]:
display(metrics_df)

,Product,R²,RMSE,MAE,Quality
0,FIC10014_PV,0.249076,0.940216,0.287906,⚠ Fair
1,FIC20017_PV,0.670056,0.552269,0.406068,⚠ Fair
2,FIC20018_PV,0.565173,0.638558,0.418536,⚠ Fair
3,FIC25021_PV,0.753116,0.480399,0.203242,✓ Good
4,FIC30018_PV,0.936412,0.245416,0.113166,⭐ Excellent
5,FI40039_PV,0.020079,1.067251,0.103692,⚠ Fair
6,FIC40045_PV,0.663105,0.571394,0.419169,⚠ Fair
7,FIC50011_PV,0.920945,0.276844,0.184530,⭐ Excellent
8,FIC60020_PV,0.896353,0.321002,0.198181,⭐ Excellent
9,FIC65004_PV,0.878577,0.346507,0.245988,⭐ Excellent


## Method 2

In [34]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


### **CELL 2: Data Preparation**
like former

### **CELL 3: Create PyTorch Dataset**

In [36]:
class MIMODataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MIMODataset(X_train, y_train)
test_dataset = MIMODataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"  DataLoader created (batch_size={batch_size}) ✓")

  DataLoader created (batch_size=32) ✓


### **CELL 4: Define Neural Network Model**

In [37]:
class MIMONetwork(nn.Module):
    def __init__(self, input_size, output_size, hidden_sizes=[256, 128, 64]):
        super(MIMONetwork, self).__init__()
        
        layers = []
        prev_size = input_size
        
        # Hidden layers
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model = MIMONetwork(
    input_size=len(utility_tags),
    output_size=len(product_tags),
    hidden_sizes=[256, 128, 64]
).to(device)

print(f"  Model Architecture:")
print(f"    Input: {len(utility_tags)} utilities")
print(f"    Hidden: 256 → 128 → 64")
print(f"    Output: {len(product_tags)} products")
print(f"    Total parameters: {sum(p.numel() for p in model.parameters())}")

  Model Architecture:
    Input: 20 utilities
    Hidden: 256 → 128 → 64
    Output: 12 products
    Total parameters: 48204


### **CELL 5: Train Neural Network**

In [39]:
import torch.optim as optim

In [40]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=False)

epochs = 150
train_losses = []
test_losses = []

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Testing
    model.eval()
    test_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            test_loss += loss.item()
    
    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    scheduler.step(test_loss)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

print("\n" + "="*120)
print("Training Complete! ✓")

Epoch  30/150 | Train Loss: 0.415063 | Test Loss: 0.366114
Epoch  60/150 | Train Loss: 0.396887 | Test Loss: 0.346659
Epoch  90/150 | Train Loss: 0.383306 | Test Loss: 0.346325
Epoch 120/150 | Train Loss: 0.373548 | Test Loss: 0.334151
Epoch 150/150 | Train Loss: 0.369979 | Test Loss: 0.343169

Training Complete! ✓


### **CELL 6: Evaluate Model**

In [41]:
model.eval()

# Predictions on all data
with torch.no_grad():
    X_all_tensor = torch.FloatTensor(X_norm).to(device)
    y_pred_norm = model(X_all_tensor).cpu().numpy()

y_actual_norm = y_norm

# Denormalize
y_pred = scaler_y.inverse_transform(y_pred_norm)
y_actual = scaler_y.inverse_transform(y_actual_norm)

# Metrics per product
metrics_list = []

print("\nPER-PRODUCT METRICS:")
print("-" * 120)

for prod_idx, prod_tag in enumerate(product_tags):
    r2 = r2_score(y_actual[:, prod_idx], y_pred[:, prod_idx])
    rmse = np.sqrt(mean_squared_error(y_actual[:, prod_idx], y_pred[:, prod_idx]))
    mae = mean_absolute_error(y_actual[:, prod_idx], y_pred[:, prod_idx])
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })
    
    print(f"{prod_tag:20} | R²: {r2:7.4f} | RMSE: {rmse:8.4f} | {quality}")

metrics_df = pd.DataFrame(metrics_list)

overall_r2 = metrics_df['R²'].mean()
print("\n" + "-" * 120)
print(f"OVERALL R² Average: {overall_r2:.4f}")


PER-PRODUCT METRICS:
------------------------------------------------------------------------------------------------------------------------
FIC10014_PV          | R²:  0.2268 | RMSE: 8025.5890 | ⚠ Fair
FIC20017_PV          | R²:  0.6580 | RMSE: 13596.0529 | ⚠ Fair
FIC20018_PV          | R²:  0.6016 | RMSE: 24621.5036 | ⚠ Fair
FIC25021_PV          | R²:  0.7895 | RMSE: 30370.9567 | ✓ Good
FIC30018_PV          | R²:  0.9019 | RMSE:  16.7961 | ⭐ Excellent
FI40039_PV           | R²:  0.1989 | RMSE: 134571294513198976030061939916800.0000 | ⚠ Fair
FIC40045_PV          | R²:  0.6585 | RMSE: 1995.7282 | ⚠ Fair
FIC50011_PV          | R²:  0.8769 | RMSE: 4254.8587 | ⭐ Excellent
FIC60020_PV          | R²:  0.8426 | RMSE: 4338.3700 | ✓ Good
FIC65004_PV          | R²:  0.8359 | RMSE: 1217.0111 | ✓ Good
FIC70053_PV          | R²:  0.8810 | RMSE: 9718.5296 | ⭐ Excellent
FIC95008_PV          | R²:  0.8570 | RMSE: 8151.9460 | ⭐ Excellent

-------------------------------------------------------------

,Product,R²,RMSE,MAE,Quality
0,FIC10014_PV,0.226804,8.025589e+03,2.566994e+03,⚠ Fair
1,FIC20017_PV,0.657996,1.359605e+04,1.001745e+04,⚠ Fair
2,FIC20018_PV,0.601552,2.462150e+04,1.637197e+04,⚠ Fair
3,FIC25021_PV,0.789532,3.037096e+04,1.369298e+04,✓ Good
4,FIC30018_PV,0.901950,1.679614e+01,7.829041e+00,⭐ Excellent
5,FI40039_PV,0.198936,1.345713e+32,1.312212e+31,⚠ Fair
6,FIC40045_PV,0.658545,1.995728e+03,1.489507e+03,⚠ Fair
7,FIC50011_PV,0.876916,4.254859e+03,2.750512e+03,⭐ Excellent
8,FIC60020_PV,0.842579,4.338370e+03,2.865229e+03,✓ Good
9,FIC65004_PV,0.835918,1.217011e+03,8.863978e+02,✓ Good


In [ ]:
display(metrics_df)